In [1]:
import pandas as pd
import pulp as plp
solver = plp.GLPK_CMD()

df = pd.read_csv('data/distances.csv', index_col=0)
df.head()

,Paris,Lyon,Marseille,Bordeaux,Nantes,Strasbourg
ville,,,,,,
Paris,0,465,775,585,385,490
Lyon,465,0,315,540,660,490
Marseille,775,315,0,640,920,800
Bordeaux,585,540,640,0,340,870
Nantes,385,660,920,340,0,830


In [2]:
villes = list(df.index)
distances = {(i, j): df.loc[i, j] for i in villes for j in villes if i != j}
n = len(villes)

In [3]:
model = plp.LpProblem('Probleme_de_TSP', plp.LpMinimize)

# Variables
x = plp.LpVariable.dicts('x', distances.keys(), cat='Binary')
u = plp.LpVariable.dicts('u', villes, lowBound = 1, upBound = n-1, cat='Integer')

# Fonction objectif
model += plp.lpSum(x[i, j]*distances[i, j] for i in villes for j in villes if i != j)

# Contraintes
for j in villes:
    model += plp.lpSum(x[i, j] for i in villes if i != j) == 1

for i in villes:
    model += plp.lpSum(x[i, j] for j in villes if i != j) == 1

depot = "Paris"
for i in villes:
    for j in villes:
        if i != j and i != depot and j != depot:
            model += u[i] - u[j] + n*x[i, j] <= n - 1 


model.solve(solver)

# Résultats
print(f"Statut : {plp.LpStatus[model.status]}")
print(f"Distance totale : {plp.value(model.objective):.0f} km\n")

print("Tour optimal :")
ville_actuelle = depot
tour = [depot]

for _ in range(n - 1):
    for j in villes:
        if ville_actuelle != j and plp.value(x[ville_actuelle, j]) == 1:
            print(f"  {ville_actuelle} → {j}")
            tour.append(j)
            ville_actuelle = j
            break

print(f"  {ville_actuelle} → {depot}")


GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --cpxlp /tmp/3948e46c7ba14548a70bda0b72096c86-pulp.lp -o /tmp/3948e46c7ba14548a70bda0b72096c86-pulp.out
 -w /tmp/3948e46c7ba14548a70bda0b72096c86-pulp.sol
Reading problem data from '/tmp/3948e46c7ba14548a70bda0b72096c86-pulp.lp'...
32 rows, 35 columns, 120 non-zeros
35 integer variables, 30 of which are binary
114 lines were read
GLPK Integer Optimizer 5.0
32 rows, 35 columns, 120 non-zeros
35 integer variables, 30 of which are binary
Preprocessing...
20 constraint coefficient(s) were reduced
32 rows, 35 columns, 120 non-zeros
35 integer variables, 30 of which are binary
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  5.000e+00  ratio =  5.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 31
Solving LP relaxation...
GLPK Simplex Optimizer 5.0
32 rows, 35 columns, 120 non-zeros
      0: obj =   3.220000000e+03 inf =   1.600e+01 (10)
     10: obj =   3.25300000